# Подготовка и первичный анализ данных для проверки бизнес-гипотез


#### __1. Формулировка бизнес-задачи__

__Контекст:__ Данные представляют собой информацию о задачах (collection), выполняемых в крупном кластере Google (Google Cluster Data). Каждая запись — это либо задача (type=5), либо набор задач (type=0,8,9). У задач есть приоритеты, ресурсные запросы и они распределяются по машинам планировщиком.

__Бизнес-задача:__ Кластер потребляет большое количество электроэнергии. Необходимо сократить эксплуатационные расходы (OPEX) за счет оптимизации использования ресурсов кластера.

#### __2. Гипотезы о факторах, влияющих на решение задачи__

- __Гипотеза 1 (Приоритет и ресурсы):__ Задачи с более высоким приоритетом (priority) имеют тенденцию запрашивать больше ресурсов (max_per_machine, max_per_switch), что может приводить к их неэффективному использованию.
- __Гипотеза 2 (Пользователи-вампиры):__ Существуют отдельные пользователи (user), которые генерируют непропорционально большое количество задач с высокими запросами ресурсов.
- __Гипотеза 3 (Планировщик и эффективность):__ Разные планировщики (scheduler) могут по-разному распределять задачи, что влияет на средний запрос ресурсов на одну задачу.
- __Гипотеза 4 (Тип коллекции):__ Многозадачные коллекции (collection_type), вероятно, имеют другие профили потребления ресурсов по сравнению с одиночными задачами.

#### __3. Определение необходимых и доступных данных__
 
 __Необходимые данные:__
- `priority`: Приоритет задачи.
- `user`: Идентификатор пользователя.
- `scheduler`: Идентификатор планировщика.
- `collection_type`: Тип коллекции.
- `max_per_machine`: Максимальное использование ресурсов на одной машине (например, CPU).
- `max_per_switch`: Максимальное использование ресурсов на коммутаторе (например, сеть или память).
- `time, collection_id, type` и другие служебные поля будут использоваться для фильтрации и идентификации записей.
 
__Доступные данные:__ Используется датасет `Dataset_500.csv`, содержащий выгрузку из системы управления кластером Google (Google Cluster Data). Данные включают информацию о задачах (`collection`), их приоритетах, запросах ресурсов, а также метаданные о пользователях и планировщиках.

##### __Номинальная шкала:__
- `type` (Тип записи: задача, набор задач и т.д.)
- `user` (Идентификатор пользователя в виде хеш-строки)
- `scheduler` (Идентификатор планировщика)
- `collection_name` (Название коллекции - хеш)
- `collection_logical_name` (Логическое имя коллекции - хеш)
- `collection_type` (Тип коллекции)

##### __Порядковая шкала:__
- `priority` (Приоритет задачи: 0, 25, 103, 115, 118, 119, 200, 360)
- `scheduling_class` (Класс планирования: 0, 1, 2, 3)
- `vertical_scaling` (Масштабирование: 1.0, 2.0, 3.0)

##### __Шкала отношений:__

###### __Исходные:__
- `time` (Временная метка события)
- `collection_id` (Уникальный идентификатор коллекции)
- `max_per_machine` (Максимальное использование ресурсов на машине)
- `max_per_switch` (Максимальное использование ресурсов на коммутаторе)

###### __Специальные форматы (требуют обработки):__
- `start_after_collection_ids` (Список ID коллекций, после которых запускается текущая)
- `missing_type` (Информация о пропущенных данных)

###### __Производные (после вычислений):__
- `has_parent_collection` (Наличие родительской коллекции - бинарный признак)
- `num_dependencies` (Количество зависимостей из `start_after_collection_ids`)
- `is_high_priority` (Высокий приоритет - бинарный признак)

###### __Трансформированные:__
- `priority_std` (стандартизированный приоритет)
- `max_per_machine_norm` (нормализованное использование ресурсов)
- `max_per_switch_std` (стандартизированное использование ресурсов на коммутаторе)
- `user_freq` (частотное кодирование пользователей)
- `scheduler_freq` (частотное кодирование планировщиков)

#### __4. Загрузка и подготовка набора данных__

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
import re
warnings.filterwarnings('ignore')

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Загрузка данных
df = pd.read_csv('Data\Dataset_500.csv', sep = ',', encoding='latin1')

In [75]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df.shape)

missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Количество пропусков': missing_data, 'Доля (%)': missing_percent})
print(missing_df)


Размер датасета: (500, 17)
                            Количество пропусков  Доля (%)
time                                           0       0.0
type                                           0       0.0
collection_id                                  0       0.0
scheduling_class                               0       0.0
collection_type                                0       0.0
priority                                       0       0.0
user                                           0       0.0
collection_name                                0       0.0
collection_logical_name                        0       0.0
start_after_collection_ids                     0       0.0
vertical_scaling                               0       0.0
scheduler                                      0       0.0
parent_collection_id                         439      87.8
missing_type                                 495      99.0
max_per_machine                              498      99.6
max_per_switch              

In [76]:
# Проверка дубликатов
print(f"Количество дубликатов: {df.duplicated().sum()}")

Количество дубликатов: 0


In [77]:
# осмотр типов данных
print(df.dtypes)

time                            int64
type                            int64
collection_id                   int64
scheduling_class                int64
collection_type                 int64
priority                        int64
user                           object
collection_name                object
collection_logical_name        object
start_after_collection_ids     object
vertical_scaling              float64
scheduler                     float64
parent_collection_id          float64
missing_type                  float64
max_per_machine               float64
max_per_switch                float64
alloc_collection_id           float64
dtype: object


#### __5. Очистка данных__

In [78]:
# копия для очистки
df_clean = df.copy()

In [79]:
# удаление дубликатов
df_clean.drop_duplicates()

,time,type,collection_id,scheduling_class,collection_type,priority,user,collection_name,collection_logical_name,start_after_collection_ids,vertical_scaling,scheduler,parent_collection_id,missing_type,max_per_machine,max_per_switch,alloc_collection_id
0,2181847986340,5,396233419682,1,0,103,/o4X3IunK+hhQOPr+NRkjWgxjdJlGtk9ujTmgsSlZKc=,uPV3/TgB5WMfJTzMEzf3tJaHuDqfgiSXFbBVl2UYWb8=,KJ8MV0Juy9uRXkIXxeA23iH+VVSuQ9++X+zRcNfUsEg=,[],2.0,1.0,NaN,NaN,NaN,NaN,NaN
1,1685777786697,5,385607584057,3,0,0,RafrUOPLtxAkG9IUoVaF8YjjXcVPt1EnEP6Zn+cQ/ko=,os7lm8wNvvr9P9qkeuA8zQY5bosEiARvh1UbtivIYG8=,bA/e9j8XWFimoE+RPjLjIgNOKj0m24eFDSE8fjQL6Ho=,[],1.0,0.0,NaN,NaN,NaN,NaN,NaN
2,65326129736,5,374545742577,2,0,115,12boQYioSPNeQSIhS/xwdE0K9pZ8Dqtfyr5X2MtxmZQ=,hiKHkc1co2UVp0OV41U2Tz3UsJIzV/ZlLBE5k/abQ4k=,mg4mwEDKc+7KvBT0lgyKDKjS8u4O75FVf32/ETjywp8=,[],1.0,1.0,NaN,NaN,NaN,NaN,NaN
3,1423212472712,5,384593615712,3,0,0,RafrUOPLtxAkG9IUoVaF8YjjXcVPt1EnEP6Zn+cQ/ko=,os7lm8wNvvr9P9qkeuA8zQY5bosEiARvh1UbtivIYG8=,bA/e9j8XWFimoE+RPjLjIgNOKj0m24eFDSE8fjQL6Ho=,[],1.0,0.0,NaN,NaN,NaN,NaN,NaN
4,669667714997,5,381064289346,2,0,115,12boQYioSPNeQSIhS/xwdE0K9pZ8Dqtfyr5X2MtxmZQ=,MoEtfRGgVrMU+I4/dOJU0lKegB3rIASUJ9evipz6Wn8=,w1Nani0WSj1xOqnvG8CuqVXorPHhlMQkHXgQYp8FZK0=,[],2.0,1.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,817007161625,5,381985330918,0,0,200,pgY0l8bIt03m9d1p0FZEmVlzW7U+GO0MzOT3ubCGkao=,fxLEDgVNK4jSlZ7wlkdri69xPdrkZC/qnuSO1jGt08I=,NTK5Vyp+44q6dIx4Q7q/6/o1pNMgljuZaybgr4+XVTY=,[],2.0,0.0,NaN,NaN,NaN,NaN,NaN
496,1107731964578,5,383272299428,2,0,119,dbLX9moZ9Oh5+ZTEbnuBY/lFFRliuhWs/l/haDAZzSQ=,Uc3/jK4i1eZS/1D3GcDoB7GOG8lDS6w3SS/ljK5SD0E=,xj8nUz2xVBUfltl4Ma69qP+X59d4L6kJwyQVygaXv7Y=,[],2.0,1.0,NaN,NaN,NaN,NaN,NaN
497,1021982137946,5,382904629886,1,0,103,XAOcLUdYZJ7Q+M7zGJLzgA2ZIXyODvLZzSngXlEFAZ0=,RS8XYQj7sipVMVXLA7Qyl76FZc2OhrialBnZk6/wCoo=,pH1xS5QwTuOGoZk0TOAInEy98fTyXmDdoU4KYZ42jAQ=,[],2.0,1.0,NaN,NaN,NaN,NaN,NaN
498,499951644165,5,380308481455,2,0,115,12boQYioSPNeQSIhS/xwdE0K9pZ8Dqtfyr5X2MtxmZQ=,MiKtumqwuuuhegfAWdZlHMxvxs3Omqu2rqtZe/VJZ0o=,hjfenpEaT0KA+RxjpMhWOZYusHN8MMGrA9fMFpkFAds=,[],2.0,1.0,NaN,NaN,NaN,NaN,NaN


In [80]:
# Удалил 5 столбцов т.к.  ничего в них нет из выборки в 500 строк
cols_to_drop = ['parent_collection_id', 'missing_type', 'max_per_machine', 'max_per_switch', 'alloc_collection_id', 'collection_type', 'start_after_collection_ids'] 
df_clean = df_clean.drop(cols_to_drop, axis=1, errors='ignore')

In [81]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df_clean.shape)

missing_data = df_clean.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Количество пропусков': missing_data, 'Доля (%)': missing_percent})
print(missing_df)


Размер датасета: (500, 10)
                         Количество пропусков  Доля (%)
time                                        0       0.0
type                                        0       0.0
collection_id                               0       0.0
scheduling_class                            0       0.0
priority                                    0       0.0
user                                        0       0.0
collection_name                             0       0.0
collection_logical_name                     0       0.0
vertical_scaling                            0       0.0
scheduler                                   0       0.0


#### __6. Масштабирование признаков__

In [82]:
# Инициализируем масштабаторы
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

df_clean['priority_std'] = scaler_standard.fit_transform(df_clean[['priority']])
df_clean['vertical_scaling_norm'] = scaler_minmax.fit_transform(df_clean[['vertical_scaling']])

In [83]:
df_clean.to_csv('Data\Dataset_final.csv', index = False, encoding='utf-8')

import pandas as pd

# Читаем исходный CSV
df = pd.read_csv('Data/Dataset.csv')

# Берем первые 500 строк
df_first_500 = df.head(500)

# Сохраняем в новый файл
df_first_500.to_csv('Data/Dataset_500.csv', index=False, encoding='utf-8')